In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import zipfile
import logging

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

from src.dataset import CustomImageDataset
from src.utils import convert_to_yolo_format



In [2]:
logging.basicConfig(
    filename="app.log",
    level=logging.INFO,
    format="%(asctime)s | %(name)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)



In [3]:
logger.info("First log in my life")



In [4]:

train_path = Path("./VisDrone2019-DET-train.zip")
test_path = Path("./VisDrone2019-DET-test-dev.zip")
val_path = Path("./VisDrone2019-DET-val.zip")
challenge_path = Path("./VisDrone2019-DET-test-challenge.zip")

data_dir = Path("./data")
data_dir.mkdir(exist_ok=True)

In [5]:


print("Extraction started")

for path_file in [train_path, challenge_path, val_path]:
    with zipfile.ZipFile(path_file, "r") as zip_ref:
        zip_ref.extractall(data_dir)

test_dir = data_dir / "VisDrone2019-DET-test-dev"
test_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(test_path, "r") as zip_ref:
    zip_ref.extractall(test_dir)

print(f"Extracted to {data_dir.resolve()}")

Extraction started
Extracted to /home/stiro/hobby/CV/Edge-Vehicle-Detection-and-Tracking/data


In [6]:
data_dir = Path("./data")

test_dataset = CustomImageDataset(data_dir / "VisDrone2019-DET-test-dev")
train_dataset = CustomImageDataset(data_dir / "VisDrone2019-DET-train")
val_dataset = CustomImageDataset(data_dir / "VisDrone2019-DET-val")
# challenge_dataset = CustomImageDataset(data_dir / "VisDrone2019-DET-test-challenge") # Do not has certain values


In [7]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True)


In [8]:
sample_path = data_dir / "VisDrone2019-DET-train" / "images" / "0000002_00005_d_0000014.jpg"


In [9]:
train_path

PosixPath('VisDrone2019-DET-train.zip')

In [10]:
CLASS_MAPPING = {
    0: "ignored_regions",
    1: "pedestrian",
    2: "people",
    3: "bicycle",
    4: "car",
    5: "van",
    6: "truck",
    7: "tricycle",
    8: "awning_tricycle",
    9: "bus",
    10: "motor",
    11: "others",
}

In [11]:
for data_set in list(data_dir.iterdir()):
    for file_path in (data_set / "annotations/").glob("*.txt"):
        convert_to_yolo_format(file_path)